# Kip Pipeline: End-to-End Claim Extraction & Verification

Run the full pipeline for **any company** from SEC data fetch to final verification results.

**Pipeline steps:**
1. Fetch SEC company facts
2. Filter SEC data by transcript years (batch process)
3. Extract claims from earnings call transcripts
4. Verify claims against SEC data

**Prerequisites:**
- Transcript `.txt` files in `data/raw/transcripts/{TICKER}/` (e.g., `2017-Jan-31-AAPL.txt`)
- `GROQ_API_KEY` in `.env` (for Step 3)
- **Note:** The verifier uses quarter→period-end mappings; these are configured for a Sep fiscal-year calendar (e.g., Apple). Other fiscal calendars may need config updates for optimal matching.

In [1]:
!pip install -r requirements.txt


[notice] A new release of pip is available: 23.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


## Configuration

Set your company and paths here.

In [8]:
import os
import sys
import subprocess
from pathlib import Path

# Project root
ROOT = Path(".").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Load .env for GROQ_API_KEY
from dotenv import load_dotenv
load_dotenv(ROOT / ".env")

TICKER = '320193' 
TICKER_ID = 'AAPL'                 
FISCAL_YEAR = 2017                  
USER_AGENT = "Kip Pipeline your.email@domain.com"  

# Fiscal year end: "dec" (Google, Microsoft) or "sep" (Apple). Affects quarter→date mapping in Step 4.
FISCAL_YEAR_END = "sep"

# Year filter: only process transcripts & financial records for this year (None = all years)
TRANSCRIPT_YEAR = 2017  # e.g., 2017 for FY2017; set to None to include all years

# Paths (defaults work for standard layout)
TRANSCRIPTS_DIR = ROOT / "data" / "raw" / "transcripts" / TICKER_ID
SEC_RAW_DIR = ROOT / "data" / "raw" / "sec"
COMPANYFACTS_PATH = SEC_RAW_DIR / "companyfacts" / f"{TICKER_ID}.json"
SEC_OUT_DIR = SEC_RAW_DIR

# Transcript filename pattern (glob)
# When TRANSCRIPT_YEAR is set: e.g. "2017-*-GOOGL.txt"; else "*-GOOGL.txt"
TRANSCRIPT_PATTERN = f"{TRANSCRIPT_YEAR}-*-{TICKER_ID}.txt" if TRANSCRIPT_YEAR else f"*-{TICKER_ID}.txt"

# Output filenames
MERGED_SEC_JSON = f"{TICKER_ID}_10K_10Q_merged_filtered_endyear.json"
CLAIMS_JSON = f"{TICKER_ID}_transcript_claims.json"
VERIFICATION_JSON = f"{TICKER_ID}_claim_verification_results.json"

print(f"Ticker: {TICKER}")
print(f"Transcripts: {TRANSCRIPTS_DIR}")
print(f"Year filter: {TRANSCRIPT_YEAR or 'all'}")
print(f"Transcript pattern: {TRANSCRIPT_PATTERN}")
print(f"Companyfacts: {COMPANYFACTS_PATH}")

Ticker: 320193
Transcripts: /Users/ashaymehta/Desktop/final kip/EarningsClaimVerifier/data/raw/transcripts/AAPL
Year filter: 2017
Transcript pattern: 2017-*-AAPL.txt
Companyfacts: /Users/ashaymehta/Desktop/final kip/EarningsClaimVerifier/data/raw/sec/companyfacts/AAPL.json


## Step 1: Fetch SEC Company Facts

Downloads the full SEC company facts JSON (skips if already cached unless `--force`).

In [9]:
def run_cmd(cmd: list, description: str = "", capture_output: bool = False):
    """Run a subprocess and print output. If capture_output=True, returns (returncode, stdout, stderr)."""
    print(f"\n>>> {description or ' '.join(cmd)}")
    result = subprocess.run(cmd, cwd=ROOT, capture_output=capture_output, text=True)
    if not capture_output and result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}")
    if capture_output:
        return result.returncode, result.stdout or "", result.stderr or ""
    return result

# Ensure companyfacts directory exists
(SEC_RAW_DIR / "companyfacts").mkdir(parents=True, exist_ok=True)

# TICKER_ID used for output filenames (fallback to TICKER if not defined)
TICKER_ID = globals().get("TICKER_ID", TICKER)

# Use TICKER to search SEC; save as TICKER_ID.json
use_cik = str(TICKER).strip().isdigit()
fetch_cmd = [
    sys.executable, "-m",
    "src.ingest.sec_fetch_companyfacts",
    "--cik" if use_cik else "--ticker", str(TICKER),
    "--out-tag", str(TICKER_ID),
    "--user-agent", USER_AGENT,
]
code, stdout, stderr = run_cmd(fetch_cmd, description="Step 1: Fetch SEC company facts", capture_output=True)
if code != 0:
    print(stderr)
    raise RuntimeError(f"Fetch failed with exit code {code}")

# File saved as TICKER_ID.json (from --out-tag)
COMPANYFACTS_PATH = SEC_RAW_DIR / "companyfacts" / f"{TICKER_ID}.json"
if not COMPANYFACTS_PATH.exists():
    raise FileNotFoundError(f"Expected {COMPANYFACTS_PATH} after fetch")
print(f"\n[OK] Company facts at {COMPANYFACTS_PATH}")


>>> Step 1: Fetch SEC company facts

[OK] Company facts at /Users/ashaymehta/Desktop/final kip/EarningsClaimVerifier/data/raw/sec/companyfacts/AAPL.json


## Step 2: Filter SEC Data by Transcript Years

Filters company facts to 10-K/10-Q data for years covered by your transcripts. When `TRANSCRIPT_YEAR` is set (e.g., 2017), only that year's transcripts and SEC data (year + prior for YoY) are used. Produces per-transcript JSONs and one merged file.

In [10]:
if not TRANSCRIPTS_DIR.exists():
    raise FileNotFoundError(
        f"Transcripts dir not found: {TRANSCRIPTS_DIR}\n"
        f"Create it and add transcript .txt files (e.g., 2017-Jan-31-{TICKER}.txt)"
    )

transcript_files = list(TRANSCRIPTS_DIR.glob(TRANSCRIPT_PATTERN))
if not transcript_files:
    raise FileNotFoundError(
        f"No transcripts matching '{TRANSCRIPT_PATTERN}' in {TRANSCRIPTS_DIR}"
    )
print(f"Found {len(transcript_files)} transcript(s)")

batch_cmd = [
    sys.executable, "-m",
    "src.data.batch_process_filter",
    "--companyfacts", str(COMPANYFACTS_PATH),
    "--transcripts-dir", str(TRANSCRIPTS_DIR),
    "--pattern", TRANSCRIPT_PATTERN,
    "--out-dir", str(SEC_OUT_DIR),
    "--include-amendments",
    "--write-merged",
    "--merged-name", MERGED_SEC_JSON,
]
if TRANSCRIPT_YEAR is not None:
    batch_cmd.extend(["--filter-year", str(TRANSCRIPT_YEAR)])

run_cmd(batch_cmd, description="Step 2: Filter SEC data by transcript years")

merged_path = SEC_OUT_DIR / MERGED_SEC_JSON
if not merged_path.exists():
    raise FileNotFoundError(f"Expected merged SEC JSON at {merged_path}")
print(f"\n[OK] Merged SEC JSON: {merged_path}")

Found 4 transcript(s)

>>> Step 2: Filter SEC data by transcript years
[OK] 2017-Aug-01-AAPL.txt -> end_years=[2015, 2016, 2017]  saved: /Users/ashaymehta/Desktop/final kip/EarningsClaimVerifier/data/raw/sec/2017-Aug-01-AAPL_10K_10Q_ENDYEAR_2015_2017.json
[OK] 2017-Jan-31-AAPL.txt -> end_years=[2015, 2016, 2017]  saved: /Users/ashaymehta/Desktop/final kip/EarningsClaimVerifier/data/raw/sec/2017-Jan-31-AAPL_10K_10Q_ENDYEAR_2015_2017.json
[OK] 2017-May-02-AAPL.txt -> end_years=[2015, 2016, 2017]  saved: /Users/ashaymehta/Desktop/final kip/EarningsClaimVerifier/data/raw/sec/2017-May-02-AAPL_10K_10Q_ENDYEAR_2015_2017.json
[OK] 2017-Nov-02-AAPL.txt -> end_years=[2015, 2016, 2017]  saved: /Users/ashaymehta/Desktop/final kip/EarningsClaimVerifier/data/raw/sec/2017-Nov-02-AAPL_10K_10Q_ENDYEAR_2015_2017.json
[OK] Merged saved: /Users/ashaymehta/Desktop/final kip/EarningsClaimVerifier/data/raw/sec/AAPL_10K_10Q_merged_filtered_endyear.json

[OK] Merged SEC JSON: /Users/ashaymehta/Desktop/final ki

## Step 3: Extract Claims from Transcripts

Uses Groq LLM to extract quantitative financial claims from each transcript. Requires `GROQ_API_KEY` in `.env`.

In [11]:
if not os.getenv("GROQ_API_KEY"):
    raise RuntimeError("Set GROQ_API_KEY in .env for claim extraction")

claims_path = SEC_OUT_DIR / CLAIMS_JSON

extract_cmd = [
    sys.executable, "-m",
    "src.transcripts.extract_claims_from_transcript",
    "--transcripts-dir", str(TRANSCRIPTS_DIR),
    "--sec-json", str(merged_path),
    "--output", str(claims_path),
    "--ticker", TICKER,
    "--fiscal-year", str(TRANSCRIPT_YEAR if TRANSCRIPT_YEAR is not None else FISCAL_YEAR),
    "--pattern", TRANSCRIPT_PATTERN,
]
if globals().get("FISCAL_YEAR_END"):
    extract_cmd.extend(["--fiscal-year-end", FISCAL_YEAR_END])
run_cmd(extract_cmd, description="Step 3: Extract claims from transcripts")

if not claims_path.exists():
    raise FileNotFoundError(f"Expected claims at {claims_path}")
print(f"\n[OK] Claims saved to {claims_path}")


>>> Step 3: Extract claims from transcripts
Loaded 277 SEC metrics from AAPL_10K_10Q_merged_filtered_endyear.json
Processing 4 transcripts from /Users/ashaymehta/Desktop/final kip/EarningsClaimVerifier/data/raw/transcripts/AAPL

[2017-Aug-01-AAPL.txt] Extracting (mapped to Q3)...
      → 67 claims extracted

[2017-Jan-31-AAPL.txt] Extracting (mapped to Q1)...
      → 47 claims extracted

[2017-May-02-AAPL.txt] Extracting (mapped to Q2)...
      → 38 claims extracted

[2017-Nov-02-AAPL.txt] Extracting (mapped to Q4)...
      → 29 claims extracted

[OK] Saved to /Users/ashaymehta/Desktop/final kip/EarningsClaimVerifier/data/raw/sec/AAPL_transcript_claims.json

[OK] Claims saved to /Users/ashaymehta/Desktop/final kip/EarningsClaimVerifier/data/raw/sec/AAPL_transcript_claims.json


## Step 4: Verify Claims Against SEC Data

Compares each extracted claim to SEC-reported values. Outputs VERIFIED / INACCURATE / UNVERIFIABLE.

In [12]:
verification_path = SEC_OUT_DIR / VERIFICATION_JSON

verify_cmd = [
    sys.executable, "-m",
    "src.verify.verify_claims_from_json",
    "--claims", str(claims_path),
    "--sec-json", str(merged_path),
    "--output", str(verification_path),
]
if globals().get("FISCAL_YEAR_END"):
    verify_cmd.extend(["--fiscal-year-end", FISCAL_YEAR_END])
run_cmd(verify_cmd, description="Step 4: Verify claims against SEC")

if not verification_path.exists():
    raise FileNotFoundError(f"Expected verification at {verification_path}")
print(f"\n[OK] Verification results: {verification_path}")


>>> Step 4: Verify claims against SEC
Saved verification results to /Users/ashaymehta/Desktop/final kip/EarningsClaimVerifier/data/raw/sec/AAPL_claim_verification_results.json

Verification complete: 181 claims
  VERIFIED:     38
  INACCURATE:   110
  UNVERIFIABLE: 33

[OK] Verification results: /Users/ashaymehta/Desktop/final kip/EarningsClaimVerifier/data/raw/sec/AAPL_claim_verification_results.json


## Final Results Summary

Load and display the verification summary.

In [13]:
import json

with open(verification_path, "r", encoding="utf-8") as f:
    results = json.load(f)

summary = results.get("summary", {})
by_verdict = summary.get("by_verdict", {})

print(f"\n{'='*50}")
print(f"  VERIFICATION RESULTS: {TICKER_ID}")
print(f"{'='*50}")
print(f"  Total claims:  {summary.get('total', 0)}")
print(f"  VERIFIED:      {by_verdict.get('VERIFIED', 0)}")
print(f"  INACCURATE:    {by_verdict.get('INACCURATE', 0)}")
print(f"  UNVERIFIABLE:  {by_verdict.get('UNVERIFIABLE', 0)}")
print(f"{'='*50}")
print(f"\nFull results: {verification_path}")


  VERIFICATION RESULTS: AAPL
  Total claims:  181
  VERIFIED:      38
  INACCURATE:    110
  UNVERIFIABLE:  33

Full results: /Users/ashaymehta/Desktop/final kip/EarningsClaimVerifier/data/raw/sec/AAPL_claim_verification_results.json


## Optional: Run Full Pipeline in One Cell

Execute all steps sequentially. Useful for batch runs or automation.

In [ ]:
# Uncomment and run to execute the entire pipeline in one go:
#
# run_cmd([sys.executable, "src/ingest/sec_fetch_companyfacts.py", "--ticker", TICKER, "--user-agent", USER_AGENT])
# run_cmd([sys.executable, "src/data/batch_process_filter.py", ...])
# run_cmd([sys.executable, "src/transcripts/extract_claims_from_transcript.py", ...])
# run_cmd([sys.executable, "src/verify/verify_claims_from_json.py", ...])
#
# Or simply run the cells above in order.